# AI Exercises – Uninformed Search

<i>
Jan Schlüter, Tara Jadidi, Tobias Morocutti, Patricia Hu, Alexander Fichtinger, Nikita Lvov
<br/>
Institute of Computational Perception, Johannes Kepler University Linz
</i>

In this notebook, you will implement a few **uninformed search** algorithms from the lecture, and apply them to two real-world problems.

## If needed: Google Colab Setup

**Only if you're running this notebook on Google Colab**, run the next cell to fetch the other files it depends on (helper modules, maps, sudokus).

**If you're running locally or on Binder**, this is not needed, and it will do nothing. Just skip it.

In [2]:
# If you're running this cell on Google Colab, it will copy in the remaining files.
# Otherwise, it doesn't do anything.

import os, sys

if 'google.colab' in sys.modules and not os.path.exists('sudoku.py'):
    if not os.path.exists('ai-exercises'):
        !git clone --quiet --branch topic01 --depth 1 https://github.com/CPJKU/ai-exercises.git
    %cd ai-exercises

## State spaces

First, we will define a general representation for a **state space** representing a problem. A state space is a graph of world **states** (for whatever limited world the problem considers) connected via **actions** that lead from one state to another.

In [7]:
"""
A world state.
"""
class State:
    pass

"""
An action in a world.
"""
class Action:
    pass

"""
A state space of states connected by actions, representing a planning problem.
"""
class Problem:

    """
    Returns the initial world state.
    """
    def get_start(self) -> State:
        ...

    """
    Checks whether a given state is a goal state.
    """
    def is_goal(self, state: State) -> bool:
        ...

    """
    For a given state, returns all possible actions and the resulting states.
    """
    def successors(self, state: State) -> list[tuple[Action, State]]:
        ...

You will not need to implement this yourself. We provide an implementation for each of the two problems we will look at. These abstract classes are just here to show by what objects and methods we represent a search problem.

## Search graphs

Secondly, we will define a way to record a *plan*, a sequence of actions and ensuing states. As in the lecture, a plan is not recorded as a list, but as a *linked list*: each execution step points at the parent node that lead to it. It also remembers the action that lead to it, and the resulting world state. While planning, if there are multiple options in a step, this will become a *tree*.

In [8]:
class Node:
    """
    A step in a plan, pointing at a world state and possibly at a preceding step and action.
    """
    def __init__(self, state: State, parent: "Node | None" = None, action: Action | None = None):
        self.state = state
        if (parent is None) ^ (action is None):
            raise ValueError("parent and action must be given or omitted together")
        self.parent = parent
        self.action = action

def expand(node: Node, problem: Problem) -> list[Node]:
    """
    Wraps all successors of a world state into search graph nodes.
    1:1 copy from the lecture, but not tracking costs yet.
    """
    successors = []
    for action, new_state in problem.successors(node.state):
        successors.append(Node(new_state, parent=node, action=action))
    return successors

To print a plan, we would follow the linked list backwards from a final state.

## First problem: Sudoku

Let us now look at the first problem: A sudoku puzzle. Each state is a (half-filled) sudoku. Each action picks one cell and fills a number. The implementation lives in [sudoku.py](./sudoku.py).

In [9]:
from sudoku import load_sudoku, draw_sudoku, draw_sudoku_tree
problem = load_sudoku('easy/1')
draw_sudoku(problem.get_start())

As you see, this is a simplified sudoku with 2x2 cells. The goal is to fill each cell, row, and column with numbers from 1 to 4.

### Random Search

We will now show a dumb search algorithm, random search. It searches the state space by picking a random successor each time.

In [10]:
import random

def random_search(problem: Problem) -> Node:
    # wrap the initial state as a search graph node
    step = Node(problem.get_start())
    # repeatedly pick a random successor
    while not problem.is_goal(step.state):
        successors = expand(step, problem)
        step = random.choice(successors)
    # we found a goal, return the linked list for the plan
    return step

To see it in action, we run it against the sudoku and draw the final state.

In [28]:
problem = load_sudoku('easy/1')
solution = random_search(problem)
draw_sudoku(solution.state, given=problem.get_start())

<span style="font-size:200%">🛑</span>
Did you get an **IndexError: Cannot choose from an empty sequence** in the `step = random.choice(successors)` line? This means it ran into a dead end with no valid options left!

### Optional detour: Repeated Random Search

A trivial fix is to have the algorithm try again from the beginning if it reaches a dead end. Skip this section if doing the notebook in class.

In [14]:
def repeated_random_search(problem: Problem) -> Node:
    num_trials = 1
    while True:
        try:
            solution = random_search(problem)
        except IndexError:
            num_trials += 1
        else:
            print(f"Took {num_trials} trials.")
            return solution

We run it against the sudoku from before:

In [19]:
problem = load_sudoku('easy/1')
solution = repeated_random_search(problem)
draw_sudoku(solution.state, given=problem.get_start())

Took 38 trials.


This was fast. Let's try a harder one.

In [29]:
problem = load_sudoku('medium/1')
solution = repeated_random_search(problem)
draw_sudoku(solution.state, given=problem.get_start())

Took 216 trials.


This was slow. It's reminiscent of [bogosort](https://en.wikipedia.org/wiki/Bogosort). We should be more systematic when reaching a dead end.

### Depth First Search

When we reach a dead end, we want to try a different solution, in a systematic way.

#### Dry run

Before you implement a better algorithm, we'll look at what it should do. If doing the notebook in class, this was already discussed; feel free to use it as a reference.

Remember that at each state, we have multiple valid options. We'll use an even simpler sudoku here:

In [32]:
problem = load_sudoku('trivial/1')
root = Node(problem.get_start())
options_1 = expand(root, problem)
draw_sudoku_tree([(root, options_1)], cellsize=30)

While Random Search always expands a random candidate to get to the next level, Depth First Search (DFS) always expands the leftmost candidate. One such expansion step:

In [33]:
options_2 = expand(options_1[0], problem)
draw_sudoku_tree([(root, options_1), (options_1[0], options_2)], cellsize=30)

Again expanding the leftmost unexpanded node in the lowest level:

In [34]:
options_3 = expand(options_2[0], problem)
draw_sudoku_tree([(root, options_1), (options_1[0], options_2), (options_2[0], options_3)], cellsize=30)

It's a dead end! It does not have any successors, since we cannot fill any valid number in the two empty cells (feel free to think it through yourself). DFS now expands the leftmost node in the lowest level that still has unexpanded nodes:

In [35]:
options_4 = expand(options_1[1], problem)
draw_sudoku_tree([(root, options_1),
                  (options_1[0], options_2),
                  (options_2[0], options_3),
                  (options_1[1], options_4)], cellsize=30)

Again going for the leftmost unexpanded node in the lowest level:

In [36]:
options_5 = expand(options_4[0], problem)
draw_sudoku_tree([(root, options_1),
                  (options_1[0], options_2),
                  (options_2[0], options_3),
                  (options_1[1], options_4),
                  (options_4[0], options_5)], cellsize=30)

And the leftmost node in the lowest level:

In [37]:
options_6 = expand(options_5[0], problem)
draw_sudoku_tree([(root, options_1),
                  (options_1[0], options_2),
                  (options_2[0], options_3),
                  (options_1[1], options_4),
                  (options_4[0], options_5),
                  (options_5[0], options_6)], cellsize=30)

We would now expand the leftmost node again. Directly before expanding a node, DFS checks whether it is a goal node. Let's also do this here, I have a good feeling:

In [38]:
problem.is_goal(options_6[0].state)

True

**Hooray**!

#### Implementation

<span style="font-size:200%">🫵</span>
It's your turn! This is a copy of the random search algorithm. Change it such that it mimics the dry run from above: At each step, instead of picking randomly, pick the leftmost unexpanded node from the lowest = **most recent** level. For this, keep all unexpanded nodes in a list, the **fringe**. Directly before expanding, check whether it is a goal. If it helps, also refer to the *Generic Tree Search Algorithm* from the lecture slides.

<div style="font-family: 'CMU Serif', 'Latin Modern Roman', Georgia, 'Times New Roman', serif; font-size: 1.05em; line-height: 1.6; margin: 1em 0;">
<b>function</b> <span style="font-variant: small-caps; color: #a31515;">Tree-Search</span>(<i>problem</i>, <i>fringe</i>) <b>returns</b> a solution, or failure
<div style="margin-left: 1.5em;">
<i>fringe</i> ← <span style="font-variant: small-caps;">Insert</span>(<span style="font-variant: small-caps;">Make-Node</span>(<span style="font-variant: small-caps;">Initial-State</span>[<i>problem</i>]), <i>fringe</i>)<br>
<b>loop do</b>
<div style="margin-left: 1.5em;">
<b>if</b> <i>fringe</i> is empty <b>then return</b> failure<br>
<i>node</i> ← <span style="font-variant: small-caps;">Remove-Front</span>(<i>fringe</i>)<br>
<b>if</b> <span style="font-variant: small-caps;">Goal-Test</span>(<i>problem</i>, <span style="font-variant: small-caps;">State</span>(<i>node</i>)) <b>then return</b> <i>node</i><br>
<i>fringe</i> ← <span style="font-variant: small-caps;">InsertAll</span>(<span style="font-variant: small-caps;">Expand</span>(<i>node</i>, <i>problem</i>), <i>fringe</i>)
</div>
</div>
</div>

In [55]:
def depth_first_search(problem: Problem) -> Node|None:

    # TODO: The following is the "Random Search" algorithm.
    # Change it! Almost nothing will remain of it.
    # - keep all unexpanded nodes in a list, the "fringe"
    # - initially, the only unexpanded node is the start node
    # - in the loop, pick the next node from the fringe
    #   - check whether it is a goal node
    #   - if not, expand it to get more unexpanded nodes
    #   - add the additional unexpanded nodes to the fringe
    # - important: think about what order to add nodes to the fringe,
    #   and what order to take nodes from the fringe, such that the
    #   algorithm always picks the leftmost unexpanded node from the
    #   lowest = most recent level

    # wrap the initial state as a search graph node
    step = Node(problem.get_start())
    fringe = [step]

    while not problem.is_goal(step.state):
      if len(fringe) == 0:
        return None
      step = fringe.pop(0)
      if problem.is_goal(step.state):
        return step
      successors = expand(step, problem)
      for successor in successors:
        problem.is_goal(successor.state)
      fringe = successors + fringe

    # # repeatedly pick a random successor
    # while not problem.is_goal(step.state):
    #     successors = expand(step, problem)
    #     step = random.choice(successors)
    # we found a goal, return the linked list for the plan
    return step

#### Verification

As a first sanity check, see whether it can solve the sudoku from the dry run, this should finish in a fraction of a second:

In [56]:
problem = load_sudoku('trivial/1')
solution = depth_first_search(problem)
draw_sudoku(solution.state, problem.get_start())

As a more thorough check, let's see whether it did the same order of expansions as the dry run. For this, we will replace the `expand()` function so it keeps a log of expansions.

In [57]:
logged_expansions = None

def expand(node: Node, problem: Problem) -> list[Node]:
    """
    Wraps all successors of a world state into search graph nodes.
    Logs each expansion in case `logged_expansions` is a list.
    """
    successors = []
    for action, new_state in problem.successors(node.state):
        successors.append(Node(new_state, parent=node, action=action))
    # logging
    global logged_expansions
    if isinstance(logged_expansions, list):
        logged_expansions.append((node, successors))
    return successors

Now solving the sudoku again and drawing the search graph as visited:

In [58]:
problem = load_sudoku('trivial/1')
global logged_expansions
logged_expansions = []
solution = depth_first_search(problem)
draw_sudoku_tree(logged_expansions, solution)

If it does not match the final tree in the dry run, your algorithm is *not* Depth First Search! Rethink in what order you add nodes to the fringe, and in what order you take them off again.

#### Harder, better, faster, stronger

If your search graph looked correct, it's time to let it loose on a harder sudoku! Stop execution when you run out of patience.

In [59]:
problem = load_sudoku('easy/1')
global logged_expansions
logged_expansions = []
solution = depth_first_search(problem)
draw_sudoku(solution.state, problem.get_start())
#draw_sudoku_tree(logged_expansions, solution)

KeyboardInterrupt: 

There are too many options to explore. Let's simplify the state space: Whenever at least *one cell* does not have any options to fill in, we already define the state as a dead end with no successors.

In [60]:
problem = load_sudoku('easy/1', early_dead_end=True)
global logged_expansions
logged_expansions = []
solution = depth_first_search(problem)
draw_sudoku(solution.state, problem.get_start())
#draw_sudoku_tree(logged_expansions, solution)

This should have worked. Let's ramp up the difficulty. Again, stop execution when you run out of patience on this one:

In [63]:
problem = load_sudoku('medium/1', early_dead_end=True)
global logged_expansions
logged_expansions = []
solution = depth_first_search(problem)
draw_sudoku(solution.state, problem.get_start())
#draw_sudoku_tree(logged_expansions, solution)

KeyboardInterrupt: 

I assume you stopped before it finished. Why is this not working? Let's look at the expansions done so far:

In [64]:
print("Expansions done:", len(logged_expansions))
print("Unique states:", len(set(node.state for node, _ in logged_expansions)))

Expansions done: 205639
Unique states: 2028


Many states are reachable via multiple sequences of actions, by filling the same fields in the same way, just in a different order. We do not want to expand states we already know, they must have lead to a dead end.

<span style="font-size:200%">🫵</span>
It's your turn! Copy your `depth_first_search()` algorithm from above, and keep track of **closed** states, the states of nodes that you already expanded. Do not expand nodes with a state you have seen before. Also feel free to refer to the extended *General Graph Search Algorithm* from the lecture.

<div style="font-family: 'CMU Serif', 'Latin Modern Roman', Georgia, 'Times New Roman', serif; font-size: 1.05em; line-height: 1.6; margin: 1em 0;">
<b>function</b> <span style="font-variant: small-caps; color: #a31515;">Graph-Search</span>(<i>problem</i>, <i>fringe</i>) <b>returns</b> a solution, or failure
<div style="margin-left: 1.5em;">
<span style="border: 1px solid red; padding: 0.1em 0.4em; margin-left: -0.4em; border-radius: 3px"><i>closed</i> ← an empty set</span><br>
<i>fringe</i> ← <span style="font-variant: small-caps;">Insert</span>(<span style="font-variant: small-caps;">Make-Node</span>(<span style="font-variant: small-caps;">Initial-State</span>[<i>problem</i>]), <i>fringe</i>)<br>
<b>loop do</b>
<div style="margin-left: 1.5em;">
<b>if</b> <i>fringe</i> is empty <b>then return</b> failure<br>
<i>node</i> ← <span style="font-variant: small-caps;">Remove-Front</span>(<i>fringe</i>)<br>
<b>if</b> <span style="font-variant: small-caps;">Goal-Test</span>(<i>problem</i>, <span style="font-variant: small-caps;">State</span>[<i>node</i>]) <b>then return</b> <i>node</i>
<div style="border: 1px solid red; padding: 0.1em 0.6em 0.1em 0.4em; margin-left: -0.4em; border-radius: 3px;">
<b>if</b> <span style="font-variant: small-caps;">State</span>[<i>node</i>] is not in <i>closed</i> <b>then</b>
<div style="margin-left: 1.5em;">
add <span style="font-variant: small-caps;">State</span>[<i>node</i>] to <i>closed</i><br>
<i>fringe</i> ← <span style="font-variant: small-caps;">InsertAll</span>(<span style="font-variant: small-caps;">Expand</span>(<i>node</i>, <i>problem</i>), <i>fringe</i>)
</div>
</div>
</div>
<b>end</b>
</div>
</div>

In [65]:
def depth_first_search(problem: Problem) -> Node|None:

    # TODO: The following is the "Random Search" algorithm.
    # Change it! Almost nothing will remain of it.
    # - keep all unexpanded nodes in a list, the "fringe"
    # - initially, the only unexpanded node is the start node
    # - in the loop, pick the next node from the fringe
    #   - check whether it is a goal node
    #   - if not, expand it to get more unexpanded nodes
    #   - add the additional unexpanded nodes to the fringe
    # - important: think about what order to add nodes to the fringe,
    #   and what order to take nodes from the fringe, such that the
    #   algorithm always picks the leftmost unexpanded node from the
    #   lowest = most recent level

    # wrap the initial state as a search graph node
    step = Node(problem.get_start())
    fringe = [step]

    closed = set()

    while not problem.is_goal(step.state):
      if len(fringe) == 0:
        return None
      step = fringe.pop(0)
      if step.state in closed:
        continue
      closed.add(step.state)
      successors = expand(step, problem)
      fringe = successors + fringe

    # # repeatedly pick a random successor
    # while not problem.is_goal(step.state):
    #     successors = expand(step, problem)
    #     step = random.choice(successors)
    # we found a goal, return the linked list for the plan
    return step

Try again:

In [66]:
problem = load_sudoku('medium/1', early_dead_end=True)
global logged_expansions
logged_expansions = []
solution = depth_first_search(problem)
draw_sudoku(solution.state, problem.get_start())

In [67]:
print("Expansions done:", len(logged_expansions))
print("Unique states:", len(set(node.state for node, _ in logged_expansions)))

Expansions done: 86975
Unique states: 86975


If it worked, you should see a correctly filled sudoku, and the number of expansions should match the number of unique states.

## Second Problem: Itinerary Planning

We will now turn to a different problem: Planning a journey in a public transport network. Each state is a potential location (stop name). Each action is to take a metro, bus or tram to a neighboring stop. The implementation lives in `journey.py`.

In [68]:
from journey import load_map, draw_map, draw_journey, print_journey, draw_journey_tree, draw_journey_expansions
vienna = load_map("vienna", restrict=["metro"])
draw_map(vienna)

### The downside of DFS

We can now run DFS to find a route from one stop to another:

In [69]:
problem = vienna.search_route('Karlsplatz', 'Westbahnhof')
global logged_expansions
logged_expansions = []
solution = depth_first_search(problem)
draw_journey(solution, vienna)

This journey does not seem particularly convenient. Let us look at the corresponding search graph, directly on the map (hover over stops to see the expansion order):

In [70]:
draw_journey_expansions(logged_expansions, vienna, solution)

Depth First Search proceeded as follows:
- explored the full U1 line North until reaching a dead end at Leopoldau
- tracked back to Praterstern and explored U2 East until the end
- tracked back to Praterstern and explored U2 West up to Museumsquartier (stopping because Karlsplatz was already expanded)
- tracked back to Volkstheater and explored U3 North until Herrengasse (because Stephansplatz was already expanded)
- tracked back to Volkstheater and explored U3 South until discovering Westbahnhof

We'd like to find a short route, not just any route.

### Breadth First Search

Instead of following every route to its maximum depth, we could advance stop by stop in each direction. This would give us a route with the fewest stops.

#### Implementation

<span style="font-size:200%">🫵</span>
It's your turn! Copy your `depth_first_search()` implementation and adapt it. Instead of expanding the leftmost candidate of the most recent additions to the fringe, always expand the *oldest* unexpanded node. This requires a very small change in how nodes are either added to the fringe or taken from the fringe. Everything else remains the same.

In [ ]:
def breadth_first_search(problem: Problem) -> Node|None:

    # TODO: Copy your last depth_first_search() implementation and adapt it:
    # - in the loop, instead of taking the latest node from the fringe,
    #   take the oldest.
    # - change how you add nodes to the fringe and/or how you remove them.

    pass

#### Verification

If correctly implemented, the search should expand stop by stop in each unexplored direction, finding a shorter route (a *shortest* route in the number of stops passed):

In [ ]:
problem = vienna.search_route('Karlsplatz', 'Westbahnhof')
global logged_expansions
logged_expansions = []
solution = breadth_first_search(problem)
draw_journey(solution, vienna)
print_journey(solution)

It should take the U2 to Volkstheater, then U3 to Westbahnhof.

And the search tree should fully expand level by level (except for stops already explored before) instead of following the leftmost candidates until a dead end:

In [ ]:
draw_journey_tree(logged_expansions, solution)

In order to find a route of 5 stops, BFS explored going 4 or 5 stops in every possible direction.

In [ ]:
draw_journey_expansions(logged_expansions, vienna, solution)

### Depth-Limited DFS and Iterative Deepening

While Breadth First Search finds the shortest path, its fringe has to store all nodes of the currently explored depth (so that it can always proceed to the next level). In Depth First Search, the fringe only stores the successors of the currently explored node and those of its ancestors up to the root. To find a shortest path with Depth First Search, we can iteratively let it search for paths up to a limited depth, until we reach a goal.

As a preparation, we adapt our `Node` class to track the depth:

In [ ]:
class Node:
    """
    A step in a plan, pointing at a world state and possibly at a preceding step and action.
    """
    def __init__(self, state: State, parent: "Node | None" = None, action: Action | None = None):
        self.state = state
        if (parent is None) ^ (action is None):
            raise ValueError("parent and action must be given or omitted together")
        self.parent = parent
        self.action = action
        self.depth = 0 if parent is None else 1 + parent.depth

#### Depth-Limited DFS

<span style="font-size:200%">🫵</span>
It's your turn! First, implement Depth-Limited DFS. Copy your DFS implementation. Adapt it to not expand a node if its depth is at the given limit. **Important:** Remove the closed set, and the check against it. (Can you see why?)

In [ ]:
def depth_limited_depth_first_search(problem: Problem, max_depth: int) -> Node|None:

    # TODO: copy your last depth_first_search() implementation
    # (not the breadth_first_search() implementation!) and adapt:
    # - before expanding a node and adding successors to the fringe,
    #   check whether node.depth < max_depth, otherwise skip expanding
    # - remove the closed set of visited states, and the check against it

    pass

With `max_depth=5`, it should find the same short path as BFS (U2 to Volkstheater, U3 to Westbahnhof):

In [ ]:
problem = vienna.search_route('Karlsplatz', 'Westbahnhof')
global logged_expansions
logged_expansions = None
solution = depth_limited_depth_first_search(problem, max_depth=5)
print_journey(solution)

While with `max_depth=4`, it should not find any solution, and return `None`:

In [ ]:
print(depth_limited_depth_first_search(problem, max_depth=4))

#### Iterative Deepening

As we usually don't know in advance in what depth we will find the shortest solution, we can just run DLDFS with `max_depth=1`, `max_depth=2`, ... until we find a goal. This way we are guaranteed to find a shortest path (as with BFS), but always have a small fringe (as it is still DFS).

<span style="font-size:200%">🫵</span>
It's your turn! Implement Iterative Deepening by iteratively calling `depth_limited_depth_first_search()` with increasing `max_depth` until it returns something else than `None`, and pass on the result. Feel free to refer to the pseudocode in the lecture slides.

In [ ]:
def iterative_deepening(problem: Problem) -> Node|None:

    # TODO: implement this from scratch
    # - iterate over max_depth
    # - call depth_limited_depth_first_search()
    # - if it is not None, return the result

    pass

Again, this should return the short route on our example itinerary:

In [ ]:
problem = vienna.search_route('Karlsplatz', 'Westbahnhof')
global logged_expansions
logged_expansions = []
solution = iterative_deepening(problem)
print_journey(solution)

And we can nicely see how it explored deeper and deeper trees (including cycles):

In [ ]:
draw_journey_tree(logged_expansions, solution)

Note that the implementation has one gotcha: If there is no solution, it will keep increasing `max_depth`, and never stop! To fix this, `depth_limited_depth_first_search()` would need a different return value for "no solution at this depth, but some options were left unexplored", and it would need a cycle check for "this state was seen already at a lower depth". We will not address it here.

### Uniform Cost Search

Up to here, we only cared about finding a route with the shortest number of stops. Instead, we might want the shortest distance traveled (the least detours), or the lowest travel time. For this, each action must be associated a *cost*. We can then define the cost of a search graph node as the sum of the action costs leading to it:

In [ ]:
class Node:
    """
    A step in a plan, pointing at a world state and possibly at a preceding step and action.
    """
    def __init__(self, state: State, parent: "Node | None" = None, action: Action | None = None):
        self.state = state
        if (parent is None) ^ (action is None):
            raise ValueError("parent and action must be given or omitted together")
        self.parent = parent
        self.action = action
        self.depth = 0 if parent is None else 1 + parent.depth
        self.cost = 0 if parent is None else getattr(action, 'cost', 0) + parent.cost

Finding an action sequence with the lowest total cost can use a strategy very similar to Breadth First Search. BFS finds a shortest path by always expanding a fringe node of the smallest depth. Uniform Cost Search (UCS) finds a cheapest path by always expanding a fringe node of the smallest cost. To efficiently select the smallest-cost node from the fringe, we make the fringe a *priority queue*: a data structure that can add in O(log(n)) and retrieve in O(1).

In [ ]:
from priority_queue import PriorityQueue

#### Implementation

<span style="font-size:200%">🫵</span>
It's your turn! Copy your `breadth_first_search()` implementation and change it to use a PriorityQueue for the fringe.

In [ ]:
def uniform_cost_search(problem: Problem) -> Node|None:

    # TODO: copy your breadth_first_search() implementation and adapt it:
    # - make the fringe a PriorityQueue()
    # - use .push(..., 0) to add the start node onto the queue with priority 0 (= cost)
    # - use .push(successor, successor.cost) for adding any other node
    # - use .pop() to remove the node with the lowest priority (= cost)

    pass

#### Shortest-distance paths

We can now try to find a route with the shortest total traveled distance, and compare it against the BFS solution. To have more options, we add trams and buses to the transit network. We define the cost of an action to be the distance traveled on the actual metro/tram/bus path between two stops, in kilometres.

In [ ]:
vienna_xxl = load_map('vienna', restrict=['metro', 'tram', 'bus'])
problem = vienna_xxl.search_route("Handelskai", "Mitte-Landstraße", cost="distance")
global logged_expansions
logged_expansions = None
solution_bfs = breadth_first_search(problem)
print(f"BFS: {solution_bfs.depth} stops, {solution_bfs.cost:.2f} km")
print_journey(solution_bfs)
draw_journey(solution_bfs, vienna_xxl, width=600)
solution_ucs = uniform_cost_search(problem)
print(f"UCS: {solution_ucs.depth} stops, {solution_ucs.cost:.2f} km")
print_journey(solution_ucs)
draw_journey(solution_ucs, vienna_xxl, width=600)

If everything was implemented correctly, BFS should find an 8-stop route of 6.54km, while UCS should find a 13-stop route of 5.09km.

#### Shortest travel time

Using the same algorithm, but a different cost definition, we can find the *fastest* route. We define the cost of an action to be the average arrival time difference of the metro/tram/bus between two particular stops, in minutes.

In [ ]:
vienna_xxl = load_map('vienna', restrict=['metro', 'tram', 'bus'])
problem = vienna_xxl.search_route("Handelskai", "Mitte-Landstraße", cost="time")
global logged_expansions
logged_expansions = None
solution_bfs = breadth_first_search(problem)
print(f"BFS: {solution_bfs.depth} stops, {solution_bfs.cost:.2f} min")
print_journey(solution_bfs)
draw_journey(solution_bfs, vienna_xxl, width=600)
solution_ucs = uniform_cost_search(problem)
print(f"UCS: {solution_ucs.depth} stops, {solution_ucs.cost:.2f} min")
print_journey(solution_ucs)
draw_journey(solution_ucs, vienna_xxl, width=600)

In this case, the route with the fewest stops is also the fastest.

### Playground

With all algorithms implemented, we finally provide a small journey planner interface for you to play with. Feel free to compare the routes with the official [WienMobil planner](https://www.wienmobil.at/en/routes).

In [ ]:
from ipywidgets import interact_manual, Combobox, Dropdown

planner_maps = {'metro': load_map('vienna', restrict=['metro']),
                'metro + tram': load_map('vienna', restrict=['metro', 'tram']),
                'metro + tram + bus': load_map('vienna', restrict=['metro', 'tram', 'bus'])}
all_stops = sorted(planner_maps['metro + tram + bus'].stops)
units = {'stops': 'stops', 'time': 'min', 'distance': 'km'}

# note: DFS and Iterative Deepening can take very long on large networks or long
# routes; use "Kernel > Interrupt" to stop them
@interact_manual(start=Combobox(options=all_stops, value='Karlsplatz', ensure_option=True),
                 goal=Combobox(options=all_stops, value='Westbahnhof', ensure_option=True),
                 network=Dropdown(options=list(planner_maps), value='metro + tram'),
                 algorithm=Dropdown(options={'Uniform Cost Search': uniform_cost_search,
                                             'Breadth First Search': breadth_first_search,
                                             'Depth First Search': depth_first_search,
                                             'Iterative Deepening': iterative_deepening}),
                 cost=Dropdown(options=['time', 'distance', 'stops']))
def plan_journey(start, goal, network, algorithm, cost):
    global logged_expansions
    logged_expansions = []
    transit_map = planner_maps[network]
    try:
        problem = transit_map.search_route(start, goal, cost=cost)
    except KeyError as e:
        print(e.args[0])
        return
    solution = algorithm(problem)
    if solution is None:
        print("No route found.")
        return
    print(f"{len(logged_expansions)} expansions, total cost: {solution.cost:.2f} {units[cost]}")
    print_journey(solution)
    draw_journey(solution, transit_map, width=700)


What's still missing: Changing from one line to another is currently free. This is left as an exercise to the reader ;) To be able to penalize it, the state would need to contain the line, not only the current station – but then we'd also end up with multiple possible starting states for a journey. The actual GTFS feed of Wiener Linien models each *platform* as a separate node with its own location. And then, the elephant in the room: schedules. To account for planned departure times of metros, trams and buses, the state would also need to include the current date and time, so waiting times can be factored in.